# 0 · One problem, four calls

Companion to **Section 2** of *The Graph Laplacian at Work*.

Everything in the paper is an instance of

$$\min_{\mathbf x}\ \tfrac12\,\|B^\top \mathbf x-\mathbf b\|_W^2,
\qquad\text{normal equations}\quad L\mathbf x = BW\mathbf b,\quad L=BWB^\top,$$

with the unknown $\mathbf x$ on the **nodes** and the data $\mathbf b$ on the **edges**.
This notebook builds the four-node graph of Example 2.4 by hand, and then
solves consensus, Tutte's drawing, sparse AHP and formation control with
**one and the same function**.

In [ ]:
import sys
sys.path.insert(0, "../code")          # the scripts that generate the paper's figures
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
np.set_printoptions(precision=4, suppress=True)


def draw(ax, P, edges, nodes_kw=None, title=None):
    """Draw a graph with node positions P (n x 2) and an edge list."""
    ax.add_collection(LineCollection([[P[i], P[j]] for i, j in edges],
                                     colors="0.75", linewidths=0.6, zorder=1))
    ax.scatter(P[:, 0], P[:, 1], s=14, zorder=2, **(nodes_kw or {}))
    ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)

## The four-node graph of Example 2.4

A triangle $v_1v_2v_3$ with a pendant node $v_4$ attached to $v_3$; the edges
are oriented as $e_1=(v_1,v_2)$, $e_2=(v_2,v_3)$, $e_3=(v_3,v_1)$, $e_4=(v_3,v_4)$.
Python counts from zero, so $v_1$ is node `0`.

In [ ]:
edges = [(0, 1), (1, 2), (2, 0), (2, 3)]
n, m = 4, len(edges)
B = np.zeros((n, m))
for h, (i, j) in enumerate(edges):
    B[i, h], B[j, h] = 1.0, -1.0       # +1 where the edge leaves, -1 where it enters
L = B @ B.T
print("B =\n", B)
print("L = B B^T =\n", L)
print("1^T B =", np.ones(n) @ B, "   L 1 =", L @ np.ones(n))

In [ ]:
from scipy.linalg import null_space
z = null_space(B).ravel()
print("ker(B) is spanned by", z / z[0], "-> the circulation around the triangle")

## Consistent and inconsistent edge data

`fit` solves the normal equations with the pseudoinverse, i.e. it returns the
solution orthogonal to $\mathbf 1$; any other solution differs by a constant.
We print $\mathbf x^\star - x^\star_3\mathbf 1$ so as to compare with the paper,
which fixes $x_3=0$.

In [ ]:
def fit(B, b, W=None):
    """Zero-mean solution of min ||B^T x - b||_W^2, and the optimal residual."""
    W = np.eye(B.shape[1]) if W is None else W
    x = np.linalg.pinv(B @ W @ B.T) @ (B @ W @ b)
    return x, b - B.T @ x

for b in (np.array([1., 1., -2., 5.]), np.array([1., 1., 1., 5.])):
    x, res = fit(B, b)
    print("b =", b, " x* - x*_3 =", x - x[2], " residual =", res,
          " sum around triangle =", b[:3].sum())

The second residual is **exactly** the circulation $\mathbf z$: no choice of
node values can explain the unit excess accumulated around the triangle,
while the pendant edge is fitted exactly (Theorem 2.3(iii)).

*Try it.* Reverse the orientation of $e_3$ and check that $L$ does not change
while $b_3$ must change sign (Exercise 1).

## One solver, four calls

The solution set of $L\mathbf x = \mathbf s$ is $\mathbf x^\star+\operatorname{span}(\mathbf 1)$.
The gradient flow $\dot{\mathbf x}=-L\mathbf x+\mathbf s$ preserves $\mathbf 1^\top\mathbf x$,
so it selects the solution with the **same mean as the initial condition**.
With anchors, stationarity in the free variables gives the grounded system
$L_{\mathcal{FF}}\mathbf x_{\mathcal F} = \mathbf s_{\mathcal F} - L_{\mathcal{FA}}\mathbf x_{\mathcal A}$.

In [ ]:
def solve_master(B, b, W=None, x0=None, anchors=None, xa=None):
    """min ||B^T x - b||_W^2 on a connected graph.

    No anchors: return the solution with the same mean as x0 (the one the
    gradient flow converges to; zero mean if x0 is None).
    Anchors: impose x[anchors] = xa and solve the grounded system.
    b may have several columns (one per coordinate, as in graph drawing).
    """
    W = np.eye(B.shape[1]) if W is None else W
    L, s = B @ W @ B.T, B @ W @ b
    if anchors is None:
        x = np.linalg.pinv(L) @ s
        return x if x0 is None else x + np.mean(x0, axis=0)
    anchors = np.asarray(anchors)
    free = np.setdiff1d(np.arange(L.shape[0]), anchors)
    x = np.zeros(s.shape)
    x[anchors] = xa
    x[free] = np.linalg.solve(L[np.ix_(free, free)],
                              s[free] - L[np.ix_(free, anchors)] @ xa)
    return x

We now call it four times on the Delaunay mesh of Section 4 (any connected
graph would do). Only the edge data and the constraint change.

In [ ]:
from common import graph_drawing, incidence
G, A, d, L, P_true, hull = graph_drawing()
B, edges = incidence(G)
n, m = B.shape
rng = np.random.default_rng(0)

# 1. average consensus: b = 0, no constraint
x0 = rng.uniform(-1, 1, n)
x_cons = solve_master(B, np.zeros(m), x0=x0)
print("1. consensus  :", x_cons[:4], "...   average of x0 =", round(x0.mean(), 4))

# 2. Tutte's drawing: b = 0 in each coordinate, hull nodes anchored
P_tutte = solve_master(B, np.zeros((m, 2)), anchors=hull, xa=P_true[hull])
print("2. Tutte      : free nodes placed, e.g. node 0 at", P_tutte[0])

# 3. sparse AHP: b = noisy log-ratios
r_true = rng.uniform(0, np.log(9), n)
b_ahp = B.T @ r_true + rng.normal(0, 0.3, m)
r_hat = solve_master(B, b_ahp, x0=r_true)          # gauge: same mean as r_true
print("3. sparse AHP : largest log-utility error =", round(np.abs(r_hat - r_true).max(), 4))

# 4. formation control: b = desired relative displacements B^T delta
k = np.arange(n)
delta = np.column_stack([np.cos(2 * np.pi * k / n), np.sin(2 * np.pi * k / n)])
p_form = solve_master(B, B.T @ delta, x0=np.zeros((n, 2)))
print("4. formation  : offsets reproduced up to a translation:",
      np.allclose(p_form - p_form.mean(0), delta - delta.mean(0)))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
draw(ax[0], P_true, edges, title="true positions")
draw(ax[1], P_tutte, edges, title="Tutte (call 2)")
ax[1].scatter(*P_true[hull].T, marker="s", color="C3", zorder=3)
plt.show()

Calls 1 and 3 are the **same computation**: consensus is sparse AHP with
$\mathbf b=\mathbf 0$. Set `b_ahp = np.zeros(m)` in call 3 and you get the
average of `r_true` back.